# PCAP CAPINFOS STATS

This file generate info and stats using the output of the `pcap_capinfos.py` script.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timedelta

In [ ]:
# Include the ndjson file created using `pcap_capinfos.py`
input_file = r'output/darknet_capinfos.ndjson'

In [ ]:
# Output files
dir_out = 'output/'
file_out_pr = dir_out + 'average_packet_rate.pdf'
file_out_ps = dir_out + 'average_packet_size.pdf'
file_out_br = dir_out + 'data_bit_rate.pdf'
file_out_fs = dir_out + 'file_size.pdf'
file_out_pc = dir_out + 'number_of_packets.pdf'
file_out_brpr = dir_out + 'double_bit_rate_packet_rate.pdf'
file_out_triplegraph = dir_out + 'triple_graph.pdf'

## Load the information

In [ ]:
# Read the filename
df = pd.read_json(input_file, lines=True)

In [ ]:
# Extract the secuence for the date (YYYYMMDD)
df["Date"] = df["file_name"].str.extract(r'(\d{8})')
df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')

In [ ]:
# Sort the dataframe by date
df = df.sort_values('Date')

## Replace and convert information

In [ ]:
# Replace " packets/s" in the 'average_packet_rate' column and convert to float
df['average_packet_rate'] = df['average_packet_rate'].str.replace(' packets/s', '').astype(float)

In [ ]:
# Convert avergage_packet_size to float
df['average_packet_size'] = df['average_packet_size'].astype(float)

In [ ]:
# Replace 'kbps' in the 'data_bit_rate' column and convert to float
df['data_bit_rate'] = df['data_bit_rate'].str.replace(' kbps', '')
df['data_bit_rate'] = df['data_bit_rate'].str.replace(' bits/s', '')
df['data_bit_rate'] = df['data_bit_rate'].astype(float)

In [ ]:
# Convert 'file_size_bytes' to float
df['file_size_bytes'] = df['file_size_bytes'].astype(float)

In [ ]:
# Convert file_size_bytes to MB
df['file_size_bytes'] = df['file_size_bytes'] / 1024  # Convert to KB
df['file_size_bytes'] = df['file_size_bytes'] / 1024  # Convert to MB

In [ ]:
# Convert 'number_of_packets' to float
df['number_of_packets'] = df['number_of_packets'].astype(int)

## Split information

We will create multiple dataframes for each statistic to plot them easily.

In [ ]:
# Create dataframes for each metric
df_pr = df[['Date', 'average_packet_rate']].copy()
df_ps = df[['Date', 'average_packet_size']].copy()
df_br = df[['Date', 'data_bit_rate']].copy()
df_fs = df[['Date', 'file_size_bytes']].copy()
df_pc = df[['Date', 'number_of_packets']].copy()

In [ ]:
# Load verifaction
print(len(df_pr))
print(len(df_ps))
print(len(df_br))
print(len(df_fs))
print(len(df_pc))

## Plot functions

In [ ]:
def dibujador_simple(df, column_name, legend_label, ylegend=None, mean_legend=None,
                     y_min=None, y_max=None, file_out=None, every_n_days=5, median_pos=None):
    sns.set(font_scale=2.0)
    sns.set_style("whitegrid")

    median_color = 'red'

    # Convert dates to matplotlib date numbers
    xnum = mdates.date2num(df['Date'])

    fig = plt.figure(figsize=(12, 7))
    plt.plot(xnum, df[column_name], label=legend_label)

    # Mean line of the values
    media_valores = df[column_name].mean()
    if mean_legend is None:
        plt.axhline(media_valores, color=median_color, linestyle='--', label=f"Mean of {legend_label.lower()}")
    else:
        plt.axhline(media_valores, color=median_color, linestyle='--', label=mean_legend)

    # Create offset for the text to avoid overlapping with the line
    if media_valores < 100:
        offset_line_sep = 1
    else:
        offset_line_sep = media_valores * 0.03  # 5% of the mean value

    # Add the value of the axhline
    if median_pos == 'left':
        plt.text(xnum[0], media_valores + offset_line_sep, f'{media_valores:.2f}', color=median_color, va='bottom', ha='left', fontsize=22)
    elif median_pos == 'right':
        plt.text(xnum[-1], media_valores + offset_line_sep, f'{media_valores:.2f}', color=median_color, va='bottom', ha='right', fontsize=22)

    # Legend
    plt.legend(
        loc='upper left',
        frameon=True,
        fancybox=True,
        borderpad=0.1,
        labelspacing=0.2,
        handlelength=1.5,
        handleheight=0.7,
        #bbox_to_anchor=(0, 1.07)
    )

    # Y-axis limits
    if y_min is not None and y_max is not None:
        plt.ylim(y_min, y_max)

    ax = plt.gca()

    # Convert numbers from matplotlib to datetime
    dates = [mdates.num2date(d) for d in xnum]

    xmin = df['Date'].min()
    xmax = df['Date'].max()

    ax.set_xlim(mdates.date2num(xmin), mdates.date2num(xmax))

    # Select one tick every every_n_days days
    days = [d for i, d in enumerate(dates) if i % every_n_days == 0]

    # Use only those days as ticks
    ax.set_xticks(days)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Date formatting
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    # Axes labels
    if ylegend:
        plt.ylabel(ylegend, fontsize=22)
    else:
        plt.ylabel(legend_label, fontsize=22)

    plt.xlabel("Date", fontsize=22)

    # Ticks X
    ax.tick_params(
        axis='x',
        direction='out',
        length=6,
        width=1,
        bottom=True,
        labelbottom=True,
        top=False
    )

    plt.tight_layout()

    # Save to file if specified
    if file_out:
        plt.savefig(file_out, format='pdf', bbox_inches='tight')

    plt.show()


In [ ]:
dibujador_simple(df_pc, column_name='number_of_packets', legend_label='Number of packets', file_out=file_out_pc, every_n_days=10, median_pos='left')

In [ ]:
dibujador_simple(df_pr, column_name='average_packet_rate', legend_label='Average Packet Rate (packets/s)', file_out=file_out_pr, every_n_days=10, median_pos='left' )

In [ ]:
dibujador_simple(df_ps, column_name='average_packet_size', legend_label='Average packet size (Bytes)',
                 mean_legend='Mean packet size', ylegend='Packet size', y_min=0, y_max=120, file_out=file_out_ps, every_n_days=10, median_pos='left')

In [ ]:
dibujador_simple(df_br, column_name='data_bit_rate', legend_label='Data bit rate (bits/s)', file_out=file_out_br, every_n_days=10, median_pos='left')

In [ ]:
dibujador_simple(df_fs, column_name='file_size_bytes', legend_label='File size (MB)', file_out=file_out_fs, every_n_days=10, median_pos='left')

In [ ]:
def dibujador_simple_barras(df, column_name, legend_label, mean_legend=None,
                            ylegend=None, y_min=None, y_max=None, file_out=None, every_n_days=5, median_pos=None):
    sns.set(font_scale=2.0)
    sns.set_style("whitegrid")

    color_mean_line = 'red'

    # Convert dates to numbers
    xnum = mdates.date2num(df['Date'])

    fig = plt.figure(figsize=(12, 7))
    plt.bar(xnum, df[column_name], label=legend_label)

    # Mean line on both axes
    media_valores = df[column_name].mean()
    if mean_legend is None:
        plt.axhline(media_valores, color=color_mean_line, linestyle='--', label=f"Mean of {legend_label.lower()}")
    else:
        plt.axhline(media_valores, color=color_mean_line, linestyle='--', label=mean_legend)

    # Add the value of the axhline on the right side
    if median_pos == 'left':
        plt.text(xnum[0], media_valores, f'{media_valores:.2f}', color=color_mean_line, va='bottom', ha='left', fontsize=22)
    elif median_pos == 'right':
        plt.text(xnum[-1], media_valores, f'{media_valores:.2f}', color=color_mean_line, va='bottom', ha='right', fontsize=22)

    # Legend
    plt.legend(
        loc='upper left',
        frameon=True,
        fancybox=True,
        borderpad=0.1,
        labelspacing=0.2,
        handlelength=1.5,
        handleheight=0.7,
        #bbox_to_anchor=(0, 1.07)
    )

    # Y-axis limits
    if y_min is None or y_max is None:
        # Get the max value of the data to set the upper limit of the y-axis
        y_min = 0
        y_max_data = df[column_name].max()
        y_offset = 1.1
        y_max = y_max_data * y_offset  # Add some space above the max value
    plt.ylim(bottom=y_min, top=y_max)

    ax = plt.gca()

    # Convert numbers from matplotlib to datetime
    dates = [mdates.num2date(d) for d in xnum]

    xmin = df['Date'].min()
    xmax = df['Date'].max()

    # Do not cut the bars
    xmin = xmin - timedelta(days=0.5)
    xmax = xmax + timedelta(days=0.5)

    ax.set_xlim(mdates.date2num(xmin), mdates.date2num(xmax))

    # Select one tick every every_n_days days
    days = [d for i, d in enumerate(dates) if i % every_n_days == 0]

    # Use only those days as ticks
    ax.set_xticks(days)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Date formatting
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    # Axes labels
    if ylegend:
        plt.ylabel(ylegend, fontsize=22)
    else:
        plt.ylabel(legend_label, fontsize=22)

    plt.xlabel("Date", fontsize=22)

    # Ticks X
    ax.tick_params(
        axis='x',
        direction='out',
        length=6,
        width=1,
        bottom=True,
        labelbottom=True,
        top=False
    )

    plt.tight_layout()

    # Save chart if needed
    if file_out:
        plt.savefig(file_out, format='pdf', bbox_inches='tight')

    plt.show()


In [ ]:
dibujador_simple_barras(df_fs, column_name='file_size_bytes', legend_label='File size (MB)',
                         mean_legend='Mean file size', ylegend='File size (MB)', y_min=None, y_max=None, file_out=file_out_fs, every_n_days=10, median_pos='left')

In [ ]:
def dibujador_doble(df1, df2, df1_col, df2_col, df1_name, df2_name, file_out, every_n_days=5):
    sns.set(font_scale=2.0)
    sns.set_style("whitegrid")

    # Create a figure and axes
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # Plot Values1 on the left Y-axis
    sns.lineplot(x='Date', y=df1_col, data=df1, ax=ax1, label='Kb/s', color='b', alpha=1) # , marker='o')

    # Create a second right Y-axis
    ax2 = ax1.twinx()

    # Plot Values2 on the right Y-axis
    sns.lineplot(x='Date', y=df2_col, data=df2, ax=ax2, label='Packets/s', color='r', alpha=1) #, marker='x')

    # Joint legend for both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    # Remove automatic legend from ax2 if it appears
    ax2.get_legend().remove() if ax2.get_legend() else None

    # Adjust axis labels
    ax1.set_xlabel('Date')
    ax1.set_ylabel(df1_name, color='b')
    ax2.set_ylabel(df2_name, color='r')

    # Set the lower limit of the right axis to 0 and the top to the maximum value of the data
    max_y_df1 = df1[df1_col].max()
    max_y_df2 = df2[df2_col].max()
    y_offset = 1.1
    ax1.set_ylim(bottom=0, top=max_y_df1 * y_offset)  # Add some space above the max value
    ax2.set_ylim(bottom=0, top=max_y_df2 * y_offset)  # Add some space above the max value

    # Convert numbers from matplotlib to datetime
    xnum = mdates.date2num(df1['Date'])
    dates = [mdates.num2date(d) for d in xnum]

    xmin = df1['Date'].min()
    xmax = df1['Date'].max()

    ax1.set_xlim(mdates.date2num(xmin), mdates.date2num(xmax))
    ax2.set_xlim(mdates.date2num(xmin), mdates.date2num(xmax))

    days = [d for i, d in enumerate(dates) if i % every_n_days == 0]

    # Use only those days as ticks
    ax1.set_xticks(days)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))  # Date formatting
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

    # Ticks X
    ax1.tick_params(
        axis='x',
        direction='out',
        length=6,
        width=1,
        bottom=True,
        labelbottom=True,
        top=False
    )

    # Adjust limits automatically
    plt.tight_layout()

    # Save this chart
    if file_out is not None:
        plt.savefig(file_out, format='pdf')

    # Show the chart
    plt.show()

dibujador_doble(df_br, df_pr, 'data_bit_rate', 'average_packet_rate', 'Kb/s', 'Packets/s', file_out=file_out_brpr, every_n_days=10)

### Mean values and other calculations to add to the article

In [ ]:
print(f"Total number of packets: {df_pc['number_of_packets'].sum()}")

In [ ]:
df_fs.sort_values('file_size_bytes')

In [ ]:
df_ps.sort_values('average_packet_size')

In [ ]:
print(f"Average packet size: {df_ps.average_packet_size.mean()}")

In [ ]:
df

In [ ]:
# This code create a LaTeX table with the statistics of the files analyzed
# The fields included are: Date, Average Packet Rate, Average Packet Size, data bit rate, File size, Number of packets
latex_table = df[['Date', 'average_packet_rate', 'average_packet_size', 'data_bit_rate', 'file_size_bytes', 'number_of_packets']].to_latex(index=False, float_format="%.2f")
print(latex_table)

In [ ]:
# Export latex table to file
with open(dir_out + 'pcap_capinfos_stats_table.tex', 'w') as f:
    f.write(latex_table)